# Household Electricity Consumption Forecasting Using Machine Learning

**Project type:** End-to-end Predictive Analytics / KDD  
**Dataset:** [Individual Household Electric Power Consumption — UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/235/individualhouseholdelectricpowerconsumption)

## Objective
Predict household active power consumption **one hour ahead** using historical electricity measurements and time-based features.

## Notes
- This notebook downloads the official UCI dataset using `ucimlrepo` if available, with a direct ZIP download fallback.
- The raw data is sampled every minute. We aggregate it to hourly observations and predict the next hour's mean `Global_active_power`.
- A chronological split is used to avoid future-data leakage.
- Metrics are reported on the held-out test period. Results are produced by running the notebook; do not claim unrun scores in the report.


## 1. Imports and configuration

In [ ]:
import sys, subprocess, importlib.util, warnings, zipfile, io, urllib.request
warnings.filterwarnings("ignore")

# Install only if a package is missing (uncomment if your environment blocks installs)
required = {"pandas": "pandas", "numpy": "numpy", "matplotlib": "matplotlib", "sklearn": "scikit-learn"}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
print("Python:", sys.version.split()[0])
print("Data directory:", DATA_DIR.resolve())


## 2. Load the UCI dataset

In [ ]:
# Try the official UCI ZIP first. The file contains semicolon-separated data.
DATA_PATH = DATA_DIR / "household_power_consumption.txt"
ZIP_PATH = DATA_DIR / "household_power_consumption.zip"
UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"

if not DATA_PATH.exists():
    try:
        print("Downloading official UCI dataset...")
        urllib.request.urlretrieve(UCI_ZIP_URL, ZIP_PATH)
        with zipfile.ZipFile(ZIP_PATH) as zf:
            candidates = [n for n in zf.namelist() if n.lower().endswith(".txt") and "household" in n.lower()]
            if not candidates:
                candidates = [n for n in zf.namelist() if n.lower().endswith(".txt")]
            if not candidates:
                raise FileNotFoundError("No .txt data file found inside the downloaded archive.")
            with zf.open(candidates[0]) as src, open(DATA_PATH, "wb") as dst:
                dst.write(src.read())
        print("Dataset saved to:", DATA_PATH)
    except Exception as e:
        print("Automatic download failed:", repr(e))
        print("Download the dataset ZIP from https://archive.ics.uci.edu/dataset/235/individualhouseholdelectricpowerconsumption")
        print("Extract household_power_consumption.txt into the ./data folder, then rerun this cell.")
        raise

raw = pd.read_csv(
    DATA_PATH,
    sep=";",
    na_values=["?"],
    low_memory=False
)
print("Raw shape:", raw.shape)
display(raw.head())
print("\nColumn types:")
display(raw.dtypes.to_frame("dtype"))


## 3. Data understanding and quality checks

In [ ]:
print("Rows, columns:", raw.shape)
print("\nMissing values:")
display(raw.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("\nDuplicate rows:", int(raw.duplicated().sum()))

# Combine Date and Time into a proper datetime index.
raw["datetime"] = pd.to_datetime(
    raw["Date"].astype(str) + " " + raw["Time"].astype(str),
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)
raw = raw.drop(columns=["Date", "Time"])
raw = raw.dropna(subset=["datetime"]).sort_values("datetime")
raw = raw.drop_duplicates(subset=["datetime"]).set_index("datetime")

numeric_cols = raw.select_dtypes(include=np.number).columns.tolist()
for col in numeric_cols:
    raw[col] = pd.to_numeric(raw[col], errors="coerce")

print("Date range:", raw.index.min(), "to", raw.index.max())
display(raw[numeric_cols].describe().T)


## 4. Preprocessing and hourly aggregation

In [ ]:
# The target is mean active power (kW) in the next hour.
TARGET_SOURCE = "Global_active_power"
if TARGET_SOURCE not in raw.columns:
    raise KeyError(f"Expected column {TARGET_SOURCE!r} not found. Available columns: {raw.columns.tolist()}")

# Convert impossible/invalid measurements to missing rather than silently treating them as real.
# Do not indiscriminately remove outliers: unusual usage may be real.
raw[numeric_cols] = raw[numeric_cols].replace([np.inf, -np.inf], np.nan)

# Resample minute-level readings into hourly mean values.
hourly = raw[numeric_cols].resample("h").mean()
hourly = hourly.rename(columns={TARGET_SOURCE: "target_power_kw"})
hourly = hourly.dropna(subset=["target_power_kw"])

print("Hourly dataset shape:", hourly.shape)
print("Hourly missing values (top 10):")
display(hourly.isna().sum().sort_values(ascending=False).head(10).to_frame("missing_count"))
display(hourly.head())


## 5. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
hourly["target_power_kw"].plot(ax=axes[0], linewidth=0.7, title="Hourly mean global active power")
axes[0].set_ylabel("Active power (kW)")
hourly["target_power_kw"].loc["2007-01":"2007-02"].plot(ax=axes[1], linewidth=0.8, title="Example period: Jan–Feb 2007")
axes[1].set_ylabel("Active power (kW)")
plt.tight_layout()
plt.show()

hourly["hour"] = hourly.index.hour
hourly["day_of_week"] = hourly.index.dayofweek
hourly["month"] = hourly.index.month
hourly["is_weekend"] = (hourly.index.dayofweek >= 5).astype(int)
hourly["year"] = hourly.index.year

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
hourly.groupby("hour")["target_power_kw"].mean().plot(kind="bar", ax=axes[0], title="Average power by hour")
hourly.groupby("day_of_week")["target_power_kw"].mean().plot(kind="bar", ax=axes[1], title="Average power by day of week")
axes[0].set_xlabel("Hour of day")
axes[1].set_xlabel("Day of week (0=Monday)")
plt.tight_layout()
plt.show()

print("Hourly target distribution:")
display(hourly["target_power_kw"].describe().to_frame("value"))


## 6. Feature engineering and forecasting target

We predict the **next hour's** mean active power. Features at time *t* are used to predict consumption at *t + 1 hour*. Lag features are created from the observed target series and rolling statistics use only past observations. The shift below is important: it makes the prediction horizon explicit.

In [ ]:
model_df = hourly.copy()

# Calendar features for the hour whose consumption is being forecast.
model_df["hour"] = model_df.index.hour
model_df["day_of_week"] = model_df.index.dayofweek
model_df["day_of_month"] = model_df.index.day
model_df["month"] = model_df.index.month
model_df["is_weekend"] = (model_df.index.dayofweek >= 5).astype(int)

# Past consumption values (at forecast origin t)
for lag in [1, 2, 3, 24, 48, 168]:
    model_df[f"lag_{lag}h"] = model_df["target_power_kw"].shift(lag)

model_df["rolling_mean_3h"] = model_df["target_power_kw"].shift(1).rolling(3).mean()
model_df["rolling_mean_24h"] = model_df["target_power_kw"].shift(1).rolling(24).mean()
model_df["rolling_std_24h"] = model_df["target_power_kw"].shift(1).rolling(24).std()

# One-hour-ahead target: consumption at t+1.
model_df["y_next_hour"] = model_df["target_power_kw"].shift(-1)

feature_cols = [
    "hour", "day_of_week", "day_of_month", "month", "is_weekend",
    "lag_1h", "lag_2h", "lag_3h", "lag_24h", "lag_48h", "lag_168h",
    "rolling_mean_3h", "rolling_mean_24h", "rolling_std_24h"
]
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + ["y_next_hour"])

X = model_df[feature_cols]
y = model_df["y_next_hour"]
print("Model matrix:", X.shape)
print("Forecast target: mean active power in the next hour (kW)")
display(model_df[feature_cols + ["y_next_hour"]].head())


## 7. Chronological train / validation / test split

In [ ]:
# Never shuffle time-series rows: train on the past, test on the future.
n = len(model_df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

print(f"Train: {len(X_train):,} rows | {X_train.index.min()} to {X_train.index.max()}")
print(f"Validation: {len(X_val):,} rows | {X_val.index.min()} to {X_val.index.max()}")
print(f"Test: {len(X_test):,} rows | {X_test.index.min()} to {X_test.index.max()}")


## 8. Baseline and model training

In [ ]:
models = {
    "Baseline (training mean)": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=150, min_samples_leaf=2, max_features=0.9,
        n_jobs=-1, random_state=RANDOM_STATE
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=150, min_samples_leaf=2, max_features=0.9,
        n_jobs=-1, random_state=RANDOM_STATE
    ),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=150, learning_rate=0.08, l2_regularization=0.1,
        random_state=RANDOM_STATE
    )
}

def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    # MAPE is unstable when actual values are near zero; report only over non-near-zero actuals.
    yt = np.asarray(y_true)
    yp = np.asarray(y_pred)
    mask = np.abs(yt) > 0.05
    mape = (np.mean(np.abs((yt[mask] - yp[mask]) / yt[mask])) * 100) if mask.any() else np.nan
    return {"MAE": mae, "RMSE": rmse, "MAPE_%_actual_gt_0.05": mape, "R2": r2_score(y_true, y_pred)}

validation_rows = []
fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    pred = model.predict(X_val)
    validation_rows.append({"Model": name, **regression_metrics(y_val, pred)})

validation_results = pd.DataFrame(validation_rows).sort_values("RMSE")
display(validation_results.style.format(precision=4))


## 9. Select model and evaluate once on the held-out test set

In [ ]:
# Choose the model with the lowest validation RMSE.
best_name = validation_results.iloc[0]["Model"]
best_model = fitted_models[best_name]
test_pred = best_model.predict(X_test)

test_results = pd.DataFrame([{"Model": best_name, **regression_metrics(y_test, test_pred)}])
print("Selected by validation RMSE:", best_name)
display(test_results.style.format(precision=4))


## 10. Results visualisation and residual analysis

In [ ]:
results = pd.DataFrame({
    "actual": y_test,
    "predicted": test_pred
}, index=y_test.index)
results["residual"] = results["actual"] - results["predicted"]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
results.iloc[:24*7][["actual", "predicted"]].plot(ax=axes[0], linewidth=1)
axes[0].set_title("Actual vs predicted: first week of test period")
axes[0].set_ylabel("Mean active power (kW)")
results["residual"].plot(ax=axes[1], linewidth=0.7, title="Test residuals (actual - predicted)")
axes[1].axhline(0, linestyle="--")
axes[1].set_ylabel("Residual (kW)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.scatter(results["actual"], results["predicted"], alpha=0.25, s=8)
lims = [
    min(results["actual"].min(), results["predicted"].min()),
    max(results["actual"].max(), results["predicted"].max())
]
plt.plot(lims, lims, linestyle="--")
plt.xlabel("Actual next-hour power (kW)")
plt.ylabel("Predicted next-hour power (kW)")
plt.title("Actual vs predicted values")
plt.tight_layout()
plt.show()


## 11. Feature importance / interpretation

In [ ]:
# Permutation importance on a chronological validation sample.
# Sample for speed while preserving the time ordering.
sample_size = min(5000, len(X_val))
X_imp = X_val.iloc[:sample_size]
y_imp = y_val.iloc[:sample_size]

try:
    perm = permutation_importance(
        best_model, X_imp, y_imp,
        scoring="neg_root_mean_squared_error",
        n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1
    )
    importance = pd.DataFrame({
        "feature": feature_cols,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std
    }).sort_values("importance_mean", ascending=False)
    display(importance)
    importance.sort_values("importance_mean").plot(
        x="feature", y="importance_mean", kind="barh", legend=False, figsize=(9, 6),
        title="Permutation importance (validation sample)"
    )
    plt.xlabel("Increase in RMSE when feature is permuted (higher = more important)")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Permutation importance could not be computed:", repr(e))


## 12. Predict the next hour from the latest available history

In [ ]:
# Use the final feature row (whose lag/rolling features only use known historical readings).
latest_X = X.iloc[[-1]]
latest_origin = latest_X.index[0]
next_hour_estimate = float(best_model.predict(latest_X)[0])
print("Latest feature timestamp:", latest_origin)
print("Forecast target hour:", latest_origin + pd.Timedelta(hours=1))
print(f"Predicted mean active power for next hour: {next_hour_estimate:.3f} kW")
print("\nNote: This is an offline demonstration using the last timestamp in the historical dataset, not a live household forecast.")


## 13. Save reproducible outputs

In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

validation_results.to_csv(OUTPUT_DIR / "validation_model_comparison.csv", index=False)
test_results.to_csv(OUTPUT_DIR / "heldout_test_metrics.csv", index=False)
results.to_csv(OUTPUT_DIR / "test_predictions.csv")
try:
    importance.to_csv(OUTPUT_DIR / "permutation_feature_importance.csv", index=False)
except NameError:
    pass

print("Saved results to:", OUTPUT_DIR.resolve())
print("Files:", [p.name for p in OUTPUT_DIR.iterdir()])


## 14. Conclusion and limitations

Complete this section **after running the notebook**:
- State which model had the lowest validation RMSE and report its held-out test metrics.
- Explain whether it improves over the mean baseline.
- Discuss the most important features using the permutation-importance output.
- Mention limitations: one household, historical data from 2006–2010, missing measurements, no external weather/occupancy data, and possible changes in consumption behaviour.
- Do not claim that the model is suitable for every household without external validation.

## References
1. Hebrail, G., & Berard, A. (2006). *Individual Household Electric Power Consumption* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C58K54
2. UCI Machine Learning Repository. Dataset page: https://archive.ics.uci.edu/dataset/235/individualhouseholdelectricpowerconsumption
